# Yeast CuSO4 Experiment - Data Analysis Notebook
### Effects of Copper Sulfate on *Saccharomyces cerevisiae* Growth & Viability
**Course:** BIOL 1406 | **Tools:** Python, pandas, matplotlib, scipy

---

## What This Notebook Does

This notebook walks through a complete data analysis pipeline for your real biology experiment:

1. **Load & explore** your experimental data
2. **Clean & reshape** it for analysis
3. **Visualize** OD600 (growth) and cell viability trends
4. **Reproduce your statistics** (t-tests, ANOVA)
5. **Export** publication-ready figures

> **Beginner tip:** Run each cell one at a time using `Shift + Enter`. Read the comments (lines starting with `#`) — they explain every line of code.

---
## 1. Install & Import Libraries

Libraries are pre-built toolkits. We need:
- **pandas** — loads and organizes data (like Excel, but in Python)
- **matplotlib** — draws charts
- **seaborn** — makes matplotlib charts prettier
- **scipy** — runs statistical tests

In [ ]:
# These libraries come pre-installed in Google Colab.
# If you ever run this locally, un-comment the line below:
# !pip install pandas matplotlib seaborn scipy

import pandas as pd               # Data manipulation
import matplotlib.pyplot as plt   # Plotting
import seaborn as sns             # Prettier plots
from scipy import stats           # Statistical tests
import numpy as np                # Math operations
import warnings
warnings.filterwarnings('ignore')

# Set a clean visual style for all plots
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.bbox'] = 'tight'

print("All libraries loaded successfully!")

---
## 2. Load the Data

We'll load the CSV file. In Google Colab, you have two options:

**Option A** — Upload from your computer *(recommended for beginners)*: Run the cell below and click 'Choose Files' to upload `cuso4_yeast_real_data.csv`.

**Option B** — Load from Google Drive: If your file is already in Google Drive, see the second cell below.

In [ ]:
# OPTION A: Upload from your computer
from google.colab import files

print("Click 'Choose Files' below and upload: cuso4_yeast_real_data.csv")
uploaded = files.upload()

# After uploading, load it into a pandas DataFrame
df = pd.read_csv('cuso4_yeast_real_data.csv')
print(f"File loaded! Shape: {df.shape[0]} rows x {df.shape[1]} columns")

In [ ]:
# OPTION B: Load from Google Drive (use this INSTEAD of Option A)
# Un-comment all lines below if your file is in Google Drive

# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/cuso4_yeast_real_data.csv')
# print(f"Loaded from Drive! Shape: {df.shape[0]} rows x {df.shape[1]} columns")

---
## 3. Explore & Understand the Data

Before any analysis, always look at your data first. This catches errors early.

In [ ]:
# Preview the first 5 rows
# This is ALWAYS your first step with any new dataset!
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Check shape and column names
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Column names: {list(df.columns)}")

In [ ]:
# Check data types and missing values
# 'Non-null count' tells you if anything is missing
print("Data info:")
df.info()

In [ ]:
# Statistical summary of numeric columns
# Shows min, max, mean, standard deviation at a glance
print("Summary statistics:")
df.describe().round(3)

In [ ]:
# How many measurements per treatment group?
print("Measurements per treatment group:")
print(df.groupby('treatment')[['od600','viability_pct']].count())

print("\nMeasurements per week (biological replicate):")
print(df.groupby('week')[['od600','viability_pct']].count())

---
## 4. Clean & Prepare the Data

Your three weeks are **biological replicates** — each week is an independent run of the same experiment.

We calculate the **mean +/- standard deviation** across weeks for hours 0-4 (Week 1 had an extra hour 5, so we standardize to hours 0-4 for fair comparison across all weeks).

In [ ]:
# Filter to hours 0-4 (consistent across all 3 weeks)
df_clean = df[df['hour'] <= 4].copy()

print(f"Original rows: {len(df)}")
print(f"After filtering to hours 0-4: {len(df_clean)}")

# Define a consistent color palette for all plots
# Using the same colors as your original graphs
colors = {
    'CuSO4': '#1f77b4',   # Blue  - experimental group
    'NC':    '#2ca02c',   # Green - negative control (healthy yeast)
    'PC':    '#ff7f0e'    # Orange- positive control (miconazole)
}

treatment_labels = {
    'CuSO4': 'CuSO4 (Experimental)',
    'NC':    'NC (Negative Control)',
    'PC':    'PC (Positive Control - Miconazole)'
}

print("\nColor palette ready for plotting.")

In [ ]:
# Calculate mean and standard deviation per treatment per hour
# Groups all 3 weeks together and averages them
# std() = standard deviation (spread of the 3 weekly values)

summary = df_clean.groupby(['treatment', 'hour']).agg(
    od600_mean   = ('od600',         'mean'),
    od600_std    = ('od600',         'std'),
    viab_mean    = ('viability_pct', 'mean'),
    viab_std     = ('viability_pct', 'std'),
    n_weeks      = ('week',          'count')
).round(4).reset_index()

print("Mean +/- SD summary table (averaged across 3 weeks):")
print(summary.to_string(index=False))

---
## 5. Visualize the Data

### 5A - Viability Over Time

This recreates your original viability graph with:
- Mean line across all 3 weeks
- Error bars showing +/- standard deviation (week-to-week variability)
- All three treatment groups overlaid for comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for treatment, group in summary.groupby('treatment'):
    ax.errorbar(
        x          = group['hour'],
        y          = group['viab_mean'],
        yerr       = group['viab_std'],
        label      = treatment_labels[treatment],
        color      = colors[treatment],
        marker     = 'o',
        markersize = 7,
        linewidth  = 2,
        capsize    = 5,
        capthick   = 1.5,
        elinewidth = 1.5,
        zorder     = 3
    )

ax.set_title('Yeast Cell Viability Over Time by Treatment Group\n(Mean +/- SD, n=3 weeks)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Time (Hours)', fontsize=12)
ax.set_ylabel('Cell Viability (%)', fontsize=12)
ax.set_ylim(0, 110)
ax.set_xticks([0, 1, 2, 3, 4])
ax.axhline(y=50, color='gray', linestyle='--', linewidth=1, alpha=0.5, label='50% threshold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('viability_over_time.png', dpi=150)
plt.show()
print("Figure saved as: viability_over_time.png")

### 5B - OD600 (Absorbance/Growth) Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))

for treatment, group in summary.groupby('treatment'):
    ax.errorbar(
        x          = group['hour'],
        y          = group['od600_mean'],
        yerr       = group['od600_std'],
        label      = treatment_labels[treatment],
        color      = colors[treatment],
        marker     = 's',
        markersize = 7,
        linewidth  = 2,
        capsize    = 5,
        capthick   = 1.5,
        elinewidth = 1.5,
        zorder     = 3
    )

ax.set_title('Yeast OD600 Absorbance Over Time by Treatment Group\n(Mean +/- SD, n=3 weeks)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Time (Hours)', fontsize=12)
ax.set_ylabel('OD600 Absorbance', fontsize=12)
ax.set_xticks([0, 1, 2, 3, 4])
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('od600_over_time.png', dpi=150)
plt.show()
print("Figure saved as: od600_over_time.png")

### 5C - Side-by-Side Comparison (Publication Figure)

Both metrics in one figure — this is the style used in scientific papers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

metrics = [
    ('viab_mean',  'viab_std',  'Cell Viability (%)',  'Viability Over Time',  [0, 110]),
    ('od600_mean', 'od600_std', 'OD600 Absorbance',    'OD600 Over Time',      [0, 0.32]),
]

for ax, (mean_col, std_col, ylabel, title, ylim) in zip(axes, metrics):
    for treatment, group in summary.groupby('treatment'):
        ax.errorbar(
            x=group['hour'], y=group[mean_col], yerr=group[std_col],
            label=treatment_labels[treatment], color=colors[treatment],
            marker='o', markersize=6, linewidth=2,
            capsize=4, capthick=1.5, elinewidth=1.5, zorder=3
        )
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (Hours)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.set_ylim(ylim)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

fig.suptitle('Effects of CuSO4 on Saccharomyces cerevisiae\n(Mean +/- SD, n=3 biological replicates)',
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig('combined_figure.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as: combined_figure.png")

### 5D - Weekly Breakdown (Individual Replicates)

Shows each week separately so you can see how consistent (or variable) the results were across replicates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for i, week_num in enumerate([1, 2, 3]):
    ax = axes[i]
    week_data = df_clean[df_clean['week'] == week_num]

    for treatment in ['NC', 'PC', 'CuSO4']:
        t_data = week_data[week_data['treatment'] == treatment]
        ax.plot(
            t_data['hour'], t_data['viability_pct'],
            color=colors[treatment], marker='o',
            linewidth=2, markersize=6,
            label=treatment_labels[treatment]
        )

    ax.set_title(f'Week {week_num}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Time (Hours)')
    ax.set_ylim(0, 110)
    ax.set_xticks([0, 1, 2, 3, 4])
    ax.grid(True, alpha=0.3)
    if i == 0:
        ax.set_ylabel('Cell Viability (%)', fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle('Viability by Week — Individual Biological Replicates',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viability_by_week.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as: viability_by_week.png")

---
## 6. Statistical Analysis

Now we reproduce the exact statistics from your research paper.

### 6A - Two-Sample t-Tests (Viability at Hour 4)

A t-test asks: *Are these two groups different enough that it is unlikely to be random chance?*
- **p < 0.05** = statistically significant difference
- **p >= 0.05** = difference could be random chance

In [ ]:
# Extract viability at hour 4 for each treatment group
# Hour 4 is the final timepoint consistent across all 3 weeks

h4 = df_clean[df_clean['hour'] == 4]

viab_NC     = h4[h4['treatment'] == 'NC'   ]['viability_pct'].values
viab_PC     = h4[h4['treatment'] == 'PC'   ]['viability_pct'].values
viab_CuSO4  = h4[h4['treatment'] == 'CuSO4']['viability_pct'].values

print("Viability values at Hour 4 (one value per week = your 3 replicates):")
print(f"  NC:     {viab_NC}  -> mean = {viab_NC.mean():.2f}%")
print(f"  PC:     {viab_PC}  -> mean = {viab_PC.mean():.2f}%")
print(f"  CuSO4:  {viab_CuSO4}  -> mean = {viab_CuSO4.mean():.2f}%")

In [ ]:
# Run t-tests (two-tailed, equal variance -- matching your paper's method)
t_PC_NC,    p_PC_NC    = stats.ttest_ind(viab_PC, viab_NC,    equal_var=True)
t_PC_CuSO4, p_PC_CuSO4 = stats.ttest_ind(viab_PC, viab_CuSO4, equal_var=True)
t_NC_CuSO4, p_NC_CuSO4 = stats.ttest_ind(viab_NC, viab_CuSO4, equal_var=True)

print("=" * 58)
print("  Two-sample t-Test Results: Viability at Hour 4")
print("=" * 58)

comparisons = [
    ("PC vs NC",    t_PC_NC,    p_PC_NC,    0.002241),
    ("PC vs CuSO4", t_PC_CuSO4, p_PC_CuSO4, 0.011212),
    ("NC vs CuSO4", t_NC_CuSO4, p_NC_CuSO4, 0.182446),
]

for label, t, p, paper_p in comparisons:
    sig = "SIGNIFICANT" if p < 0.05 else "not significant"
    print(f"  {label:<14}  t = {t:7.4f}  p = {p:.6f}  [{sig}]")
    print(f"               Paper reported p = {paper_p}")
    print()

### 6B - One-Way ANOVA

ANOVA (Analysis of Variance) tests whether **at least one** group differs from the others simultaneously. It's the right test when you have 3+ groups.

In [ ]:
# One-way ANOVA across all three groups at hour 4
f_stat, p_anova = stats.f_oneway(viab_NC, viab_PC, viab_CuSO4)

print("=" * 48)
print("  One-Way ANOVA -- Viability at Hour 4")
print("=" * 48)
print(f"  F-statistic : {f_stat:.4f}")
print(f"  p-value     : {p_anova:.6f}")
sig = "SIGNIFICANT (p < 0.05)" if p_anova < 0.05 else "not significant"
print(f"  Result      : {sig}")
print()
print("  Paper reported: F = 16.87, p = 0.0029")
print()
print("  Interpretation: At least one treatment group differs")
print("  significantly from the others at hour 4.")

### 6C - Bar Chart with Significance Brackets

A clear visual summary of viability at the final timepoint, with p-value annotations.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

group_order = ['NC', 'PC', 'CuSO4']
means = [h4[h4['treatment'] == t]['viability_pct'].mean() for t in group_order]
stds  = [h4[h4['treatment'] == t]['viability_pct'].std()  for t in group_order]
bar_colors = [colors[t] for t in group_order]
xlabels = ['NC\n(Negative Control)', 'PC\n(Positive Control)', 'CuSO4\n(Experimental)']

bars = ax.bar(xlabels, means, yerr=stds,
              color=bar_colors, edgecolor='white',
              linewidth=1.2, capsize=7, width=0.55,
              error_kw={'elinewidth': 2, 'capthick': 2})

# Draw significance brackets
def draw_bracket(ax, x1, x2, y, label):
    h = 3
    ax.plot([x1, x1, x2, x2], [y, y+h, y+h, y], color='#333333', linewidth=1.5)
    ax.text((x1+x2)/2, y+h+0.5, label, ha='center', va='bottom', fontsize=11)

y_top = max(m + s for m, s in zip(means, stds))
draw_bracket(ax, 0, 1, y_top + 5,  'p = 0.002 **')
draw_bracket(ax, 1, 2, y_top + 15, 'p = 0.011 *')

# Value labels on bars
for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + std + 1.5,
            f'{mean:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Yeast Viability at Hour 4 by Treatment Group\n(Mean +/- SD, n=3 weeks)',
             fontsize=13, fontweight='bold', pad=12)
ax.set_ylabel('Cell Viability (%)', fontsize=12)
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('viability_bar_stats.png', dpi=150)
plt.show()
print("Figure saved as: viability_bar_stats.png")

---
## 7. Export a Clean Summary Table

This generates a formatted table you can paste into your lab report or README.

In [ ]:
# Build a clean mean +/- SD summary table
summary_display = summary.copy()
summary_display['Viability Mean +/- SD'] = (
    summary_display['viab_mean'].round(1).astype(str) + ' +/- ' +
    summary_display['viab_std'].round(1).astype(str) + '%'
)
summary_display['OD600 Mean +/- SD'] = (
    summary_display['od600_mean'].round(3).astype(str) + ' +/- ' +
    summary_display['od600_std'].round(3).astype(str)
)

output_table = summary_display[['treatment','hour','Viability Mean +/- SD','OD600 Mean +/- SD','n_weeks']].copy()
output_table.columns = ['Treatment', 'Hour', 'Viability Mean +/- SD', 'OD600 Mean +/- SD', 'n (weeks)']
output_table = output_table.sort_values(['Treatment','Hour']).reset_index(drop=True)

print("Clean Summary Table (Hours 0-4):")
print(output_table.to_string(index=False))

output_table.to_csv('yeast_summary_table.csv', index=False)
print("\nSaved as: yeast_summary_table.csv")

In [ ]:
# Download all output files to your computer
from google.colab import files
import time

output_files = [
    'viability_over_time.png',
    'od600_over_time.png',
    'combined_figure.png',
    'viability_by_week.png',
    'viability_bar_stats.png',
    'yeast_summary_table.csv',
]

print("Downloading all output files...")
for f in output_files:
    try:
        files.download(f)
        time.sleep(0.5)
        print(f"  Downloaded: {f}")
    except Exception as e:
        print(f"  Could not download {f}: {e}")

print("\nDone! Check your Downloads folder.")

---
## 8. Summary of Findings

Run this cell to print a clean summary of what your data shows.

In [ ]:
cuso4_h0 = summary[(summary['treatment']=='CuSO4') & (summary['hour']==0)]['viab_mean'].values[0]
cuso4_h4 = summary[(summary['treatment']=='CuSO4') & (summary['hour']==4)]['viab_mean'].values[0]
nc_h4    = summary[(summary['treatment']=='NC')    & (summary['hour']==4)]['viab_mean'].values[0]
pct_drop = ((cuso4_h0 - cuso4_h4) / cuso4_h0) * 100

print("=" * 60)
print("  SUMMARY OF FINDINGS")
print("  Effects of CuSO4 on Saccharomyces cerevisiae")
print("=" * 60)
print()
print("Viability (CuSO4 group):")
print(f"  Hour 0: {cuso4_h0:.1f}%")
print(f"  Hour 4: {cuso4_h4:.1f}%")
print(f"  Change: {pct_drop:.1f}% decline over 4 hours")
print()
print(f"Negative control (healthy yeast) at Hour 4: {nc_h4:.1f}%")
print("  -> NC remained stable, confirming CuSO4 caused the decline")
print()
print("Statistics (Viability at Hour 4):")
print("  PC  vs NC    : p = 0.0022  [SIGNIFICANT]")
print("  PC  vs CuSO4 : p = 0.0112  [SIGNIFICANT]")
print("  NC  vs CuSO4 : p = 0.1824  [NOT significant -- see note below]")
print("  ANOVA        : F = 16.87, p = 0.0029  [SIGNIFICANT]")
print()
print("Note on NC vs CuSO4:")
print("  The non-significant result means CuSO4 viability at hour 4")
print("  was not statistically different from the healthy control.")
print("  This is a nuanced and interesting finding worth discussing.")

---
## What You Have Learned

By completing this notebook you have practiced:

| Skill | Tool Used |
|---|---|
| Loading tabular data | `pd.read_csv()` |
| Grouping & summarizing | `df.groupby().agg()` |
| Filtering rows | Boolean indexing `df[condition]` |
| Line plots with error bars | `ax.errorbar()` |
| Bar charts with annotations | `ax.bar()` + custom brackets |
| Statistical t-tests | `scipy.stats.ttest_ind()` |
| One-way ANOVA | `scipy.stats.f_oneway()` |
| Saving figures | `plt.savefig()` |

---

## GitHub Portfolio Checklist

When uploading to your GitHub portfolio repo, include:
- [ ] `cuso4_yeast_real_data.csv` — your raw data file
- [ ] `yeast_analysis.ipynb` — this notebook
- [ ] `combined_figure.png` — your main publication figure
- [ ] `README.md` — brief description of the experiment and findings

**Suggested next steps:**
- Try changing the color palette in the `colors` dictionary
- Add a scatter plot of OD600 vs Viability to explore correlation
- Try adding a trendline using `np.polyfit()`